# Construye tu primer agente de IA con LangChain

**Taller práctico · EurusConf 2026**

Este notebook es la **Parte 1** del taller y corre en **Google Colab** (no instalas nada en tu máquina).

Vas a construir un **agente investigador** y luego lo vas a hacer **sólido**: que se vea bien (streaming), que recuerde (memoria) y que devuelva datos limpios (salida estructurada).

> En la **Parte 2** pasamos a VS Code y llevamos este mismo agente a un **proyecto real** (con `uv` + `src/`). Eso vive en el repo.

**Cómo correrlo:** ve celda por celda (Shift+Enter). Corre primero el Setup.

## 0. Setup

In [ ]:
# Instala las librerías (en Colab). Tarda ~1 min.
%pip install -q "langchain>=1.0" langchain-google-genai langchain-openai langchain-community ddgs python-dotenv pydantic requests

In [ ]:
# Carga tu API key y prepara el modelo (funciona en Colab y en VS Code)
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # lee .env si existe (VS Code)

PROVIDER = os.getenv("MODEL_PROVIDER", "google_genai")
MODEL_NAME = os.getenv("MODEL_NAME", "gemini-2.0-flash")

# En Colab te pide la key aquí (gratis en https://aistudio.google.com/apikey)
if PROVIDER == "google_genai" and not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Pega tu GOOGLE_API_KEY: ")

def get_model():
    if PROVIDER == "openrouter":  # OpenRouter usa la API compatible con OpenAI
        return init_chat_model(f"openai:{MODEL_NAME}",
                               base_url="https://openrouter.ai/api/v1",
                               api_key=os.getenv("OPENROUTER_API_KEY"))
    return init_chat_model(f"{PROVIDER}:{MODEL_NAME}")

model = get_model()
print("Modelo listo:", PROVIDER, MODEL_NAME)

In [ ]:
# Smoke test: si esto responde, estás listo para el taller
print(model.invoke("Responde en una sola frase: ¿qué es un agente de IA?").content)

## Recordatorio rápido

Ya vimos la teoría en las diapositivas. Lo esencial para programar:

- Un **agente** = modelo + herramientas + un *loop*: recibe una tarea, decide si usar una herramienta, observa el resultado y repite hasta responder.
- Hoy lo armamos con **`create_agent`** (de LangChain), que por debajo corre sobre **LangGraph**.
- Una **herramienta (`@tool`)** es solo una función de Python con una descripción. El modelo decide cuándo llamarla.

## Actividad 1 — Tu agente investigador

Primero un agente con una herramienta trivial (para ver el *loop*), y luego el de verdad: uno que **busca en la web**.

In [ ]:
# DEMO 1 — calentamiento: un agente con UNA herramienta simple, para ver el loop
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def hora_actual(zona: str = "local") -> str:
    """Devuelve la fecha y hora actual."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M")

agente = create_agent(
    model=model,
    tools=[hora_actual],
    system_prompt="Eres un asistente útil. Usa las herramientas cuando ayuden.",
)
r = agente.invoke({"messages": [{"role": "user", "content": "¿Qué fecha y hora es?"}]})
print(r["messages"][-1].content)

In [ ]:
# DEMO 2 — el agente investigador: una herramienta que lo conecta a internet
from langchain_community.tools import DuckDuckGoSearchRun

buscar_web = DuckDuckGoSearchRun()

investigador = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres un investigador. Busca en la web y responde con información actual y clara.",
)
r = investigador.invoke({"messages": [{"role": "user", "content": "¿Qué es LangChain y para qué se usa?"}]})
print(r["messages"][-1].content)

### Tu turno
Enfoca el investigador a **TU** tema (tu carrera, un hobby) cambiando el `system_prompt`, y hazle una pregunta actual.

In [ ]:
# TODO: crea tu investigador con un system_prompt enfocado a tu tema
# mi_investigador = create_agent(model=model, tools=[buscar_web], system_prompt="...")
# r = mi_investigador.invoke({"messages": [{"role": "user", "content": "..."}]})
# print(r["messages"][-1].content)

### Solución A1

In [ ]:
mi_investigador = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres un investigador experto en tecnología. Responde claro y con datos actuales.",
)
r = mi_investigador.invoke({"messages": [{"role": "user", "content": "¿Qué novedades hay en inteligencia artificial?"}]})
print(r["messages"][-1].content)

## Actividad 2 — Hazlo sólido

El mismo agente, pero mejor: **streaming** (se ve la respuesta llegar), **memoria** (recuerda la conversación) y **salida estructurada** (devuelve datos limpios, no texto suelto).

### 2.1 Streaming — que se vea la respuesta llegar en vivo

In [ ]:
# En vez de esperar toda la respuesta, la imprimimos token por token
from langchain_core.messages import AIMessageChunk

pregunta = {"messages": [{"role": "user", "content": "En 3 frases, ¿por qué importan los agentes de IA?"}]}
for chunk, meta in investigador.stream(pregunta, stream_mode="messages"):
    if isinstance(chunk, AIMessageChunk) and chunk.content:
        print(chunk.content, end="", flush=True)
print()

### 2.2 Memoria — que recuerde la conversación
Con un `checkpointer` y un `thread_id`, el agente recuerda lo que ya hablaron.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

investigador_memoria = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres un investigador que recuerda la conversación.",
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "demo-1"}}
investigador_memoria.invoke({"messages": [{"role": "user", "content": "Me interesan los agentes de IA."}]}, config)
r = investigador_memoria.invoke({"messages": [{"role": "user", "content": "¿Qué tema te dije que me interesa?"}]}, config)
print(r["messages"][-1].content)  # debería recordarlo

### 2.3 Salida estructurada — datos limpios en vez de texto suelto
Útil para integrar con otros sistemas (guardar en una base, mandar a una API...).

In [ ]:
from pydantic import BaseModel, Field

class Resumen(BaseModel):
    tema: str = Field(description="el tema consultado")
    puntos_clave: list[str] = Field(description="3 a 5 puntos clave, en frases cortas")

estructurado = model.with_structured_output(Resumen)
print(estructurado.invoke("Resume en puntos clave qué es LangChain"))

### Tu turno
Dale **memoria** a tu investigador (con tu propio `thread_id`) y compruébalo en dos turnos.

In [ ]:
# TODO: crea tu investigador con checkpointer=InMemorySaver(), define un config con tu thread_id,
# invócalo dos veces y comprueba que recuerda algo del primer turno.

### Solución A2

In [ ]:
mi_agente = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres mi asistente de investigación. Recuerda lo que te digo.",
    checkpointer=InMemorySaver(),
)
cfg = {"configurable": {"thread_id": "mi-sesion"}}
mi_agente.invoke({"messages": [{"role": "user", "content": "Estoy investigando sobre energías renovables."}]}, cfg)
r = mi_agente.invoke({"messages": [{"role": "user", "content": "Dame una noticia reciente sobre lo que estoy investigando."}]}, cfg)
print(r["messages"][-1].content)

## ¿Por qué construir tu propio agente si Claude Code ya existe?

Para tareas generales, usa lo que ya existe. Construyes el **tuyo** cuando tienes un flujo específico y repetible que se beneficia de:

- **Control:** tú decides qué herramientas tiene y qué puede hacer.
- **Integración:** se conecta a **tus** sistemas (tu base de datos, tu API, tu Notion).
- **Costo y escala:** corre desatendido con el modelo más barato que sirva.
- **Automatización real:** se dispara por eventos, no es un chat.

## Parte 2 — del notebook al proyecto real (en VS Code)

Hasta aquí, todo en el notebook. Ahora llevamos **este mismo agente** a un **proyecto real** con la estructura que usa LangChain:

```
src/agente/
  agent.py      # create_agent(model, tools, system_prompt, checkpointer)
  tools.py      # las @tool
  prompts.py    # el system prompt
main.py         # punto de entrada (agent.invoke)
.env  ·  pyproject.toml  ·  uv.lock
```

Lo hacemos juntos en el repo con **`uv`** (el gestor de paquetes moderno). El paso a paso está en el **README**.

### Si te sobra tiempo: 3 ideas para tu propio agente
Construye uno tuyo en ese proyecto, conectado a algo real (todas con token, sin OAuth):

1. **Investigador + scraping** — busca y además extrae info de una página.
2. **Tu productividad** — Notion o Todoist: convierte lo que escribes en tareas/notas.
3. **Datos en vivo** — una API pública (clima, finanzas, noticias) + memoria.

El código de ejemplo de cada idea está en el **README** del repo.

---

**Recursos:** https://docs.langchain.com · **MART Automations:** https://martautomations.com